# 02A — Dataset integrity & structure


In [30]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [31]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📋 var_id uniqueness


In [32]:
summary = pd.Series({
    'rows': len(d),
    'unique_var_id': d['var_id'].nunique(dropna=True),
    'duplicate_var_id_rows': int(d['var_id'].duplicated(keep=False).sum())
})
display(summary.to_frame('value'))
display(d.loc[d['var_id'].duplicated(keep=False), ['var_id', 'chr', 'pos', 'ref', 'alt']].head(20))


,value
rows,11308
unique_var_id,11308
duplicate_var_id_rows,0


,var_id,chr,pos,ref,alt


## 2. 📋 rsid uniqueness


In [33]:
rs = d.loc[~d['missing_rsid'], 'rsid'].astype(str)
summary = pd.Series({
    'non_missing_rsid_rows': int(len(rs)),
    'unique_rsid': int(rs.nunique()),
    'duplicated_rsid_rows': int(rs.duplicated(keep=False).sum())
})
display(summary.to_frame('value'))


,value
non_missing_rsid_rows,7979
unique_rsid,6900
duplicated_rsid_rows,2059


## 3. 📊 Missing rsid fraction


In [34]:
miss_share = d['missing_rsid'].mean()
out = pd.Series({'missing_rsid_fraction': miss_share, 'missing_rsid_percent': miss_share * 100})
display(out.to_frame('value'))
plot_df = out.rename('value').reset_index().rename(columns={'index': 'metric'})
fig = px.bar(plot_df, x='metric', y='value', title='Missing rsid share')
fig.show()


,value
missing_rsid_fraction,0.294393
missing_rsid_percent,29.439335


## 4. 📊 Distribution variants per rsid


In [35]:
rs_counts = d.loc[~d['missing_rsid']].groupby('rsid').size().reset_index(name='variant_count')
display(rs_counts['variant_count'].describe().to_frame('value'))
fig = px.histogram(rs_counts, x='variant_count', nbins=50, title='Variants per rsid')
fig.show()


,value
count,6900.000000
mean,1.156377
std,0.402606
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,5.000000


## 5. 📋 Duplicated variants table


In [36]:
dup_cols = ['chr', 'pos', 'ref', 'alt', 'var_type']
dup_table = d.groupby(dup_cols, dropna=False).size().reset_index(name='n').query('n > 1').sort_values('n', ascending=False)
display(dup_table.head(50))


,chr,pos,ref,alt,var_type,n
29,X,168546.0,NaN,NaN,copy number loss,20
8,X,10701.0,NaN,NaN,copy number loss,16
4037,X,32305626.0,NaN,NaN,Deletion,14
300,X,31140036.0,NaN,NaN,Duplication,14
3889,X,32235023.0,NaN,NaN,Deletion,14
3887,X,32235013.0,NaN,NaN,Deletion,10
2626,X,31676097.0,NaN,NaN,Deletion,10
4039,X,32305636.0,NaN,NaN,Deletion,9
3213,X,31854815.0,NaN,NaN,Deletion,9
3888,X,32235013.0,NaN,NaN,Duplication,9


## 6. 📊 Missingness heatmap by key columns


In [37]:
key_cols = ['rsid', 'ref', 'alt', 'clinvar_consequence', 'domain', 'exon', 'interval_length', 'revel', 'meta_lr', 'allele_freq']
miss = d[key_cols].isna().mean().to_frame('missing_fraction')
fig = px.imshow(miss.T, aspect='auto', color_continuous_scale='Reds', title='Missingness heatmap: key columns')
fig.show()


## 7. 📋 Coverage (exon, domain, frame_status)


In [38]:
cov_cols = ['exon', 'domain', 'frame_status']
coverage = pd.DataFrame({
    'column': cov_cols,
    'non_missing_n': [int(d[c].notna().sum()) for c in cov_cols],
    'coverage_percent': [round(d[c].notna().mean() * 100, 2) for c in cov_cols]
})
display(coverage)


,column,non_missing_n,coverage_percent
0,exon,11098,98.14
1,domain,3205,28.34
2,frame_status,11308,100.00


## 8. 📊 Missingness vs mutation_type_group


In [39]:
tmp = d.groupby('mutation_type').agg(
    missing_domain=('domain', lambda s: s.isna().mean()),
    missing_exon=('exon', lambda s: s.isna().mean()),
    missing_revel=('revel_num', lambda s: s.isna().mean()),
    n=('var_id', 'size')
).reset_index().sort_values('n', ascending=False)
display(tmp)
fig = px.bar(tmp.melt(id_vars=['mutation_type', 'n'], value_vars=['missing_domain', 'missing_exon', 'missing_revel']), x='mutation_type', y='value', color='variable', barmode='group', title='Missingness by mutation_type_group')
fig.show()


,mutation_type,missing_domain,missing_exon,missing_revel,n
2,missense,0.409856,0.000000,0.344050,3328
1,large_deletion,0.998824,0.108760,1.000000,1701
4,splice,0.859574,0.002128,0.904255,940
3,nonsense,0.072061,0.000000,1.000000,791
0,frameshift,0.535809,0.000000,1.000000,754


## 9. 🧪 Fisher: missing domain ~ mutation_type


In [40]:
rows = []
for mt in sorted(d['mutation_type'].dropna().unique()):
    tmp = d[['mutation_type', 'missing_domain']].dropna().copy()
    tmp['is_mt'] = tmp['mutation_type'].eq(mt)
    tab, odds, p, ci = fisher_bool(tmp, 'is_mt', 'missing_domain')
    rows.append({'mutation_type': mt, 'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]})
res = pd.DataFrame(rows).sort_values('p_value')
display(res)


,mutation_type,odds_ratio,p_value,or_ci_low,or_ci_high
1,large_deletion,1025.981770,0.000000e+00,256.133598,4109.724763
3,nonsense,0.044469,6.614564e-222,0.033810,0.058488
2,missense,0.285007,2.972774e-151,0.258913,0.313732
4,splice,5.297871,2.813758e-88,4.379888,6.408254
0,frameshift,0.832214,1.778063e-02,0.715534,0.967921


## 10. 📊 Missingness vs interval_length


In [41]:
tmp = d[['missing_domain', 'interval_length_num']].dropna().copy()
fig = px.box(tmp, x='missing_domain', y='interval_length_num', points='outliers', title='Missing domain vs interval_length')
fig.update_yaxes(type='log')
fig.show()


## 11. 🧪 Mann–Whitney: missing domain vs interval_length


In [42]:
g1, g0, stat, p = mann_whitney_bool(d, 'missing_domain', 'interval_length_num')
out = pd.Series({'n_missing_domain': len(g1), 'n_domain_present': len(g0), 'u_statistic': stat, 'p_value': p})
display(out.to_frame('value'))


,value
n_missing_domain,2.317000e+03
n_domain_present,2.000000e+02
u_statistic,3.835035e+05
p_value,4.996205e-54


## 12. 📋 Consistency chr-pos (sanity table)


In [43]:
chr_clean = d['chr'].astype(str).str.upper().str.replace('CHR', '', regex=False)
san = pd.DataFrame([
    {'check': 'chromosome_is_X', 'n_fail': int((chr_clean != 'X').sum())},
    {'check': 'pos_not_missing', 'n_fail': int(d['pos_num'].isna().sum())},
    {'check': 'pos_positive', 'n_fail': int((d['pos_num'] <= 0).fillna(False).sum())}
])
display(san)
issues = d.loc[(chr_clean != 'X') | d['pos_num'].isna() | (d['pos_num'] <= 0), ['var_id', 'chr', 'pos', 'ref', 'alt']]
display(issues.head(20))


,check,n_fail
0,chromosome_is_X,0
1,pos_not_missing,0
2,pos_positive,0


,var_id,chr,pos,ref,alt
